In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define our scope (5 States, 4 Districts each)
states_districts = {
    "Uttar Pradesh": ["Varanasi", "Gorakhpur", "Agra", "Aligarh"],
    "Maharashtra": ["Pune", "Nashik", "Satara", "Solapur"],
    "Bihar": ["Patna", "Gaya", "Muzaffarpur", "Bhagalpur"],
    "Tamil Nadu": ["Madurai", "Salem", "Vellore", "Erode"],
    "Karnataka": ["Mysore", "Hubli", "Belagavi", "Bellary"]
}

years = [2022, 2023, 2024, 2025]
villages_per_district = 125 # 20 districts * 125 villages * 4 years = 10,000 rows

data_rows = []
village_id_counter = 10001

# Generate the Base Village-Level Data
for state, districts in states_districts.items():
    for district in districts:
        for v in range(villages_per_district):
            village_name = f"{district}_Village_{v+1}"
            for year in years:
                data_rows.append([village_id_counter, village_name, district, state, year])
            village_id_counter += 1

base_df = pd.DataFrame(data_rows, columns=["Village_ID", "Village_Name", "District", "State", "Year"])

# ---------------------------------------------------------
# 1. Rural Employment Dataset
# ---------------------------------------------------------
emp_df = base_df.copy()
# Village level numbers are smaller than district level
emp_df["MGNREGA_Workers"] = np.random.randint(50, 300, size=len(emp_df))
emp_df["Skill_Dev_Participants"] = np.random.randint(5, 40, size=len(emp_df))
emp_df["Avg_Daily_Wage_INR"] = np.random.randint(200, 350, size=len(emp_df)) + ((emp_df["Year"] - 2022) * 15)
emp_df["Unemployment_Rate_Pct"] = np.round(np.random.uniform(4.5, 15.0, size=len(emp_df)) - ((emp_df["Year"] - 2022) * 0.4), 1)
emp_df["Unemployment_Rate_Pct"] = emp_df["Unemployment_Rate_Pct"].clip(lower=1.0) # Can't go below 1%

# ---------------------------------------------------------
# 2. Agricultural Productivity Dataset
# ---------------------------------------------------------
agr_df = base_df[['Village_ID', 'Year']].copy()
agr_df["Irrigated_Area_Hectares"] = np.random.randint(100, 800, size=len(agr_df))
agr_df["Crop_Yield_MT_per_Hec"] = np.round(np.random.uniform(1.5, 4.5, size=len(agr_df)) + ((agr_df["Year"] - 2022) * 0.15), 2)
agr_df["Fertilizer_Usage_Tons"] = np.random.randint(10, 100, size=len(agr_df))
agr_df["Tractor_Availability"] = np.random.randint(1, 15, size=len(agr_df))

# ---------------------------------------------------------
# 3. Healthcare & Education (Social) Dataset
# ---------------------------------------------------------
soc_df = base_df[['Village_ID', 'Year']].copy()
# Some villages might have 0 or 1 school/clinic
soc_df["Primary_Schools"] = np.random.randint(0, 4, size=len(soc_df))
soc_df["School_Dropout_Rate_Pct"] = np.round(np.random.uniform(5.0, 25.0, size=len(soc_df)) - ((soc_df["Year"] - 2022) * 1.5), 1)
soc_df["School_Dropout_Rate_Pct"] = soc_df["School_Dropout_Rate_Pct"].clip(lower=0.0)
soc_df["Primary_Health_Centres"] = np.random.randint(0, 2, size=len(soc_df))
soc_df["Clean_Water_Access_Pct"] = np.round(np.random.uniform(30.0, 90.0, size=len(soc_df)) + ((soc_df["Year"] - 2022) * 3.0), 1)
soc_df["Clean_Water_Access_Pct"] = soc_df["Clean_Water_Access_Pct"].clip(upper=100.0)

# ---------------------------------------------------------
# 4. Infrastructure Growth Dataset
# ---------------------------------------------------------
inf_df = base_df[['Village_ID', 'Year']].copy()
inf_df["Electrification_Pct"] = np.round(np.random.uniform(50.0, 98.0, size=len(inf_df)) + ((inf_df["Year"] - 2022) * 2.5), 1)
inf_df["Internet_Penetration_Pct"] = np.round(np.random.uniform(10.0, 60.0, size=len(inf_df)) + ((inf_df["Year"] - 2022) * 6.0), 1)
inf_df["Paved_Roads_KM"] = np.round(np.random.uniform(2.0, 20.0, size=len(inf_df)), 1)
# Most villages won't have a bank branch, maybe 1 if lucky
inf_df["Bank_Branches"] = np.random.choice([0, 0, 0, 1, 2], size=len(inf_df))

inf_df["Electrification_Pct"] = inf_df["Electrification_Pct"].clip(upper=100.0)
inf_df["Internet_Penetration_Pct"] = inf_df["Internet_Penetration_Pct"].clip(upper=100.0)

# ---------------------------------------------------------
# Save to CSV
# ---------------------------------------------------------
emp_df.to_csv("1_Rural_Employment_10k.csv", index=False)
agr_df.to_csv("2_Agriculture_10k.csv", index=False)
soc_df.to_csv("3_Social_Indicators_10k.csv", index=False)
inf_df.to_csv("4_Infrastructure_10k.csv", index=False)

print(f"✅ Success! Generated {len(emp_df)} rows across all 4 datasets.")

✅ Success! Generated 10000 rows across all 4 datasets.


In [2]:
import pandas as pd

# 1. Load the 4 generated CSV files
emp = pd.read_csv("1_Rural_Employment_10k.csv")
agr = pd.read_csv("2_Agriculture_10k.csv")
soc = pd.read_csv("3_Social_Indicators_10k.csv")
inf = pd.read_csv("4_Infrastructure_10k.csv")

# 2. Merge them together based on Village_ID and Year
# This does exactly what Power Query would do!
master = emp.merge(agr, on=["Village_ID", "Year"])
master = master.merge(soc, on=["Village_ID", "Year"])
master = master.merge(inf, on=["Village_ID", "Year"])

# 3. Save the final Master file
master.to_csv("Master_Rural_Data.csv", index=False)

print("✅ Merge Complete! 'Master_Rural_Data.csv' is ready for WPS Office.")

✅ Merge Complete! 'Master_Rural_Data.csv' is ready for WPS Office.
